In [ ]:
# install dependencies
!pip install torch torchvision gradio -q
!pip install gradio gdown -q

# test imports
import torch
import torchvision
import gradio
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay # for confusion matrix
import gradio as gr # for gradio demo

In [ ]:
import os

# Paste your Kaggle API token string here
KAGGLE_TOKEN = ""

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

# Download dataset
!pip install kaggle -q
!kaggle datasets download -d uzairkhan45/game-sub-categorized-android-apps-screenshots
!unzip -q game-sub-categorized-android-apps-screenshots.zip -d /content/game_data/

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class_names = ['GAME_ACTION', 'GAME_ADVENTURE', 'GAME_PUZZLE',
               'GAME_RACING', 'GAME_SPORTS', 'GAME_STRATEGY']

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_base = '/content/game_data/Game_Sub_Categorized_Android_Apps_Screenshots/train'
val_base   = '/content/game_data/Game_Sub_Categorized_Android_Apps_Screenshots/validation'

# Map selected class names to new indices 0-5
class_to_new_label = {cls: i for i, cls in enumerate(class_names)}

# Custom dataset that skips corrupted images
class SafeImageFolder(ImageFolder):
    def __getitem__(self, index):
        try:
            return super().__getitem__(index)
        except (UnidentifiedImageError, OSError):
            # Skip bad image, return next valid one
            return self.__getitem__((index + 1) % len(self))

full_train = SafeImageFolder(train_base, transform=transform_train)
full_val   = SafeImageFolder(val_base,   transform=transform_test)

# Filter samples and remap labels
def filter_and_remap(dataset):
    new_samples = []
    for path, label in dataset.samples:
        cls = dataset.classes[label]
        if cls in class_to_new_label:
            new_samples.append((path, class_to_new_label[cls]))
    dataset.samples = new_samples
    dataset.targets = [s[1] for s in new_samples]
    dataset.classes = class_names
    dataset.class_to_idx = class_to_new_label
    return dataset

train_dataset = filter_and_remap(full_train)
val_dataset   = filter_and_remap(full_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {class_names}")

In [ ]:
import torchvision.models as models
import torch.nn as nn

# load pretrained ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# replace the final layer with one matching number of classes
num_classes = len(class_names)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# move model to GPU
model = model.to(device)

# train all layers with a small learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

print("Model is ready.")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# training loop
train_accs, val_accs, losses = [], [], []

num_epochs = 5

for epoch in range(num_epochs):
    # training
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = 100. * correct / total
    avg_loss = running_loss / len(train_loader)

    # validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100. * val_correct / val_total
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Val Acc: {val_acc:.2f}%")

    train_accs.append(train_acc)
    val_accs.append(val_acc)
    losses.append(avg_loss)

print("\nTraining complete!")

In [ ]:
# training curves
epochs = range(1, num_epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_accs, label='Train Accuracy')
ax1.plot(epochs, val_accs, label='Val Accuracy')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy (%)')
ax1.legend()

ax2.plot(epochs, losses, color='red', label='Train Loss')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# confusion matrix
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion Matrix — ResNet-18 on Game Visuals")
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print("\nPer-class accuracy:")
for i, cls in enumerate(class_names):
    cls_acc = cm[i, i] / cm[i].sum() * 100
    print(f"  {cls:<12}: {cls_acc:.1f}%")

In [ ]:
# gradio demo
def predict(image):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img_tensor = transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1).squeeze().cpu().numpy()
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=5),
    title="Video Game Genre Classifier",
    description="Upload a game screenshot to predict its genre."
)
demo.launch(share=True)